# 15 — Coverage-aware task redesign

This milestone audits class/family coverage from 03–14, reuses the 13 trajectory-level splits, and evaluates a calibrated 120 s family classifier with a validation-only unknown/reject threshold. Test data are used only for final reporting.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from coverage_task_redesign import run_experiment

result = run_experiment(project_root=PROJECT_ROOT, result_root=PROJECT_ROOT / 'results')
summary = result['summary']
print(json.dumps({
    'cohort_count': summary['cohort_count'],
    'coverage_tiers': summary['coverage_tiers'],
    'comparison_test_summary': summary['comparison_test_summary'],
    'rejection_summary': summary['rejection_summary'],
}, ensure_ascii=False, indent=2, default=str))

In [ ]:
required = [
    '15_class_task_coverage.csv',
    '15_family_task_coverage.csv',
    '15_selective_metrics.csv',
    '15_calibration_metrics.csv',
    '15_rejection_analysis.csv',
    '15_npp_guard_v1_capability_matrix.csv',
    '15_npp_guard_v1_capability_matrix.json',
    '15_summary.json',
]
figures = [
    '15_risk_coverage.png',
    '15_macro_f1_vs_coverage.png',
    '15_reliability_diagram.png',
    '15_forced_vs_selective.png',
    '15_rejection_focus_classes.png',
]
assert summary['cohort_count'] == 505
assert all(summary['assertions'].values())
assert all((PROJECT_ROOT / 'results' / name).exists() for name in required)
assert all((PROJECT_ROOT / 'results' / 'figures' / name).exists() for name in figures)
assert not result['class_coverage'].empty and not result['family_coverage'].empty
assert set(result['class_coverage']['status']).issubset({'supported', 'exploratory', 'not-supported'})
assert set(result['family_coverage']['status']).issubset({'supported', 'exploratory', 'not-supported'})
comparison = pd.DataFrame(summary['comparison_test_summary'])
assert {'forced_12_class', 'forced_family', 'family_selective'}.issubset(set(comparison['mode']))
print('FULL COVERAGE-AWARE TASK REDESIGN PASSED')
display(result['class_coverage'][['class', 'family', 'tier', 'status', 'matched_trajectories']])
display(result['family_coverage'][['family', 'tier', 'status', 'matched_trajectories']])
display(comparison)